In [53]:
import torch
import pickle
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import selfies as sf
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, DataStructs, rdmolops, Descriptors, rdMolDescriptors, QED
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import math
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
from sklearn.metrics import r2_score
from IPython.display import display
from torch_geometric.data import Data
import seaborn as sns
import os
import re
torch.cuda.empty_cache()

In [12]:
df = pd.read_csv('/Users/mateusz/Documents/apki/Graph-Latent-Space/data/raw/smiles_selfies_full.csv')

In [13]:
df.head()

,smiles,selfies
0,O=S(O)c1cc2c(cc1F)OC(c1ccc(F)cc1F)(c1ccc(F)cc1...,[O][=S][Branch1][C][O][C][=C][C][=C][Branch1][...
1,CN(C)Cc1cccc(C2Nc3cccc4c(=O)[nH]nc(c34)C2c2ccc...,[C][N][Branch1][C][C][C][C][=C][C][=C][C][Bran...
2,O=C(N[C@@H](CO)c1nc2cc(Cl)ccc2[nH]1)c1ccc(C(=O...,[O][=C][Branch2][Ring1][#Branch1][N][C@@H1][Br...
3,O=C(Cn1cc(I)cn1)N1CCCc2c1cnn2-c1ccc(F)cc1,[O][=C][Branch1][N][C][N][C][=C][Branch1][C][I...
4,Cc1ccc(-c2ccnc(Cl)c2)n1CC(=O)OCc1ccccc1,[C][C][=C][C][=C][Branch1][N][C][=C][C][=N][C]...


In [44]:
# convert to graphs
SUPPORTED_ATOMS = [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]

def atom_to_feature_vector(atom):
    """zamienia atom na wektor cech, ten wektor będzie potem nodem w grafie"""
    atomic_num = atom.GetAtomicNum()
    # one hot encoding
    return [1 if atomic_num == atom_type else 0 for atom_type in SUPPORTED_ATOMS]

def smiles_to_graph(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return None
    
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append(atom_to_feature_vector(atom))
    x = torch.tensor(atom_features, dtype=torch.float)

    edges = []
    for bond in mol.GetBonds():
        edges.append((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
        edges.append((bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()))
    
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    data = Data(x=x, edge_index=edge_index)
    return data
    

In [49]:
smiles = "CC(CC)O" 
graph = smiles_to_graph(smiles)
print(graph)
print(graph.x)
print(graph.edge_index)

Data(x=[5, 10], edge_index=[2, 8])
tensor([[0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]])
tensor([[0, 1, 1, 2, 2, 3, 1, 4],
        [1, 0, 2, 1, 3, 2, 4, 1]])


In [ ]:
GRAPH_PATH = '/Users/mateusz/Documents/apki/Graph-Latent-Space/data/processed/graphs.pt'

if os.path.exists(GRAPH_PATH):
    graph_list = torch.load(GRAPH_PATH)
else:
    graph_list = []
    for index, row in tqdm(df.iterrows(), total=len(df)):
        smiles = row['smiles']
        graph = smiles_to_graph(smiles)
        graph_list.append(graph)
    torch.save(graph_list, GRAPH_PATH)

In [ ]:
# split dataset
graph_train, graph_temp = train_test_split(graph_list, test_size=0.2, random_state=42, shuffle=True)
graph_val, graph_test = train_test_split(graph_temp, test_size=0.5, random_state=42, shuffle=True)

class GraphDataset(Dataset):
    def __init__(self, graph_list):
        self.graphs = graph_list

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, index):
        return self.graphs[index]

train_dataset = GraphDataset(graph_train)
val_dataset = GraphDataset(graph_val)
test_dataset = GraphDataset(graph_test)